In [59]:
from pathlib import Path

def find_project_root():
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / '.git').exists() or (parent / 'data').exists():
            return parent
    return current

PROJECT_ROOT = find_project_root()
DATA_EXTERNAL = PROJECT_ROOT / 'data' / 'external'

print(f"Check: Proyecto raíz: {PROJECT_ROOT}")
print(f"Check: Datos external: {DATA_EXTERNAL}")

Check: Proyecto raíz: c:\Users\alefe\OneDrive\Documentos\GitHub\data-analytics-project
Check: Datos external: c:\Users\alefe\OneDrive\Documentos\GitHub\data-analytics-project\data\external


---

## 1 — El modelo mental 

- ¿Qué divide pandas? (Split) **Divide partes del dataframe en grupos**
- ¿Qué aplica pandas a cada grupo? (Apply)
    **Transformacion de datos u operaciones**
- ¿Qué devuelve al final? (Combine)
    **Resultado de operaciones sobre grupos**
---

---

## 1 — Ventas totales por región

Se calcula el total facturado por `Region` para identificar dónde se concentra el volumen comercial.

---

In [ ]:
import pandas as pd
df = pd.read_csv(DATA_EXTERNAL / 'train.csv')

print('------------------------------')
print('Canidad de registros y columnas')
print(df.shape)
print('------------------------------')
print('Primeros registros y sus títulos')
print(df.head())
print('------------------------------')
print('Total facturado por región')
print('__________________________')
total_facturado_region = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
print(total_facturado_region)
print('__________________________')



------------------------------
Canidad de registros y columnas
(9800, 18)
------------------------------
Primeros registros y sus títulos
   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2017-152156  08/11/2017  11/11/2017    Second Class    CG-12520   
1       2  CA-2017-152156  08/11/2017  11/11/2017    Second Class    CG-12520   
2       3  CA-2017-138688  12/06/2017  16/06/2017    Second Class    DV-13045   
3       4  US-2016-108966  11/10/2016  18/10/2016  Standard Class    SO-20335   
4       5  US-2016-108966  11/10/2016  18/10/2016  Standard Class    SO-20335   

     Customer Name    Segment        Country             City       State  \
0      Claire Gute   Consumer  United States        Henderson    Kentucky   
1      Claire Gute   Consumer  United States        Henderson    Kentucky   
2  Darrin Van Huff  Corporate  United States      Los Angeles  California   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale     F


---

## 2 — Serie vs DataFrame en groupby

Las dos sintaxis producen resultados distintos:

- **Versión A:** `df.groupby('Category')['Sales'].sum()` → devuelve una **Serie**
- **Versión B:** `df.groupby('Category')[['Sales']].sum()` → devuelve un **DataFrame**

La diferencia importa cuando se quieren añadir más columnas, hacer un merge o exportar. La versión A es más rápida de leer; la versión B es más segura para operaciones posteriores.

---

In [61]:
print(df.groupby("Category")["Sales"].sum())

print(df.groupby("Category")[["Sales"]].sum())


Category
Furniture          728658.5757
Office Supplies    705422.3340
Technology         827455.8730
Name: Sales, dtype: float64
                       Sales
Category                    
Furniture        728658.5757
Office Supplies  705422.3340
Technology       827455.8730



---

## 3 — Tres métricas por segmento

Se obtiene un resumen por `Segment` (Consumer, Corporate, Home Office): ventas totales, venta media por pedido y número de pedidos.

---

In [62]:
#Calculo de las tres métricas — cada una en su propia línea, no todas juntas.
print('------------------------------')
print('Total facturado por región')
ventas_totales = df.groupby('Segment')[['Sales']].sum()
print(ventas_totales)
print('------------------------------')
print('Venta media por pedido')
venta_media = df.groupby('Segment')['Sales'].mean()
print(venta_media)
print('------------------------------')
print('Número de pedidos')
num_pedidos = df.groupby('Segment')['Order ID'].count()
print(num_pedidos)
print('------------------------------')



------------------------------
Total facturado por región
                    Sales
Segment                  
Consumer     1.148061e+06
Corporate    6.884941e+05
Home Office  4.249822e+05
------------------------------
Venta media por pedido
Segment
Consumer       225.065777
Corporate      233.150720
Home Office    243.403309
Name: Sales, dtype: float64
------------------------------
Número de pedidos
Segment
Consumer       5101
Corporate      2953
Home Office    1746
Name: Order ID, dtype: int64
------------------------------



---

## 4 — Comparación entre categorías

Se identifica qué `Category` lidera en ventas totales y cuál está última. Se agrupa, ordena de mayor a menor y se extrae la primera y última posición.

---

In [63]:
print('------------------------------')
print('Categoría por ventas')
categoria_ventas = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
print(categoria_ventas)
# Extraer primera y última del resultado ya ordenado
print('------------------------------')
categoria_max = categoria_ventas.idxmax()
categoria_min = categoria_ventas.idxmin()
print('Categoría que más vende:', categoria_max)
print('Categoría que menos vende:', categoria_min)
print('------------------------------')

# Diferencia entre primera y última
gap = categoria_ventas.max() - categoria_ventas.min()
print(f'Diferencia entre primera y última: ${gap:,.2f}')
print('------------------------------')
print('Si se divide en %, cada departamento equivale a:')
porcentajes = (categoria_ventas / categoria_ventas.sum() * 100).round(1)
print(porcentajes)



------------------------------
Categoría por ventas
Category
Technology         827455.8730
Furniture          728658.5757
Office Supplies    705422.3340
Name: Sales, dtype: float64
------------------------------
Categoría que más vende: Technology
Categoría que menos vende: Office Supplies
------------------------------
Diferencia entre primera y última: $122,033.54
------------------------------
Si se divide en %, cada departamento equivale a:
Category
Technology         36.6
Furniture          32.2
Office Supplies    31.2
Name: Sales, dtype: float64



---

## 5 — `reset_index()` y su utilidad para merges

Se obtiene el ranking de regiones por ventas totales como DataFrame, no como Serie con índice. `reset_index()` convierte la columna agrupada (que quedó como índice) en columna real — necesario para cualquier merge o exportación posterior.

---

In [64]:
print('------------------------------')
print('Total facturado por región')
print('__________________________')
total_facturado_region = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
print(total_facturado_region)
print('__________________________')
# Por qué importa en la empresa:
# Cuando hagas un merge entre dos tablas necesitas que las columnas a unir sean columnas reales. 
# Si intentas hacer pd.merge(df1, df2, on="Region") y Region es el índice en vez de una columna, 
# pandas no la encuentra y falla.

# reset_index() convierte el índice en columna para que el resultado sea usable en 
# cualquier operación posterior — merge, export a CSV, filtrado, lo que sea.

# Resumen en una línea: groupby convierte la columna agrupada en índice. reset_index() la devuelve a ser columna.
resultado_reseteado = total_facturado_region.reset_index()
print('El resultado del reset_index es: \n',  resultado_reseteado)
print('__________________________')

------------------------------
Total facturado por región
__________________________
Region
West       710219.6845
East       669518.7260
Central    492646.9132
South      389151.4590
Name: Sales, dtype: float64
__________________________
El resultado del reset_index es: 
     Region        Sales
0     West  710219.6845
1     East  669518.7260
2  Central  492646.9132
3    South  389151.4590
__________________________



---

## 6 — Insight final

Se traduce el resultado numérico en un hallazgo de negocio siguiendo la cadena **Observación → Patrón → Implicación**:

> Technology lidera con $827,456 en ventas, representando el 36.6% del total. Furniture y Office Supplies se sitúan cerca, con 32.2% y 31.2% respectivamente — una diferencia de solo $122,000 entre la primera y la última categoría. El patrón muestra una cartera equilibrada: ninguna categoría concentra más de un tercio del negocio, lo que reduce la exposición al riesgo. Sin embargo, Technology es el motor principal y cualquier caída en esa categoría impactaría desproporcionadamente los ingresos.
